# Fine-tuning SecBERT LM for text classification
Mount your own drive space as working space with the following three commands



Upload into the secBert Directory in your drive space dataset's files

In [ ]:
# !ls /var/cuda-repo-9-0-local | grep .pub
# !apt-key add /var/cuda-repo-9-0-local/7fa2af80.pub
# !apt-get update
# !sudo apt-get install cuda-9.0
# !nvcc --version
# !nvidia-smi

In [ ]:
# !pip install pandas
# !pip3 install torch torchvision
# !pip install transformers
# !pip install sklearn

In [ ]:
import torch
import pandas as pd

from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModelForMaskedLM, BertConfig, AutoModel
from sklearn.preprocessing import LabelEncoder

from time import sleep

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("jackaduma/SecBERT")

pretrained_model = AutoModelForMaskedLM.from_pretrained("jackaduma/SecBERT")
config = BertConfig.from_pretrained("jackaduma/SecBERT", output_hidden_states=True)

Set the Runtime to GPU and check and set cuda availability with the following snippet

In [ ]:
# Setting up the device for GPU usage

from torch import cuda
device = 'cuda' if cuda.is_available() else 'cpu'
device

Substitute dataset file name with your own

In [ ]:
df = pd.read_csv('./dataset.csv')
df = df.reset_index()

LABELS = len(df['label_tec'].value_counts())

#Encoding labels
encoder = LabelEncoder()
encoder.fit(df['label_tec'])
print("Number of labels in encoder:", len(encoder.classes_))
print("Label classes:", encoder.classes_)
df['enc_label'] = encoder.transform(df['label_tec'])

df = df[['label_tec', 'sentence', 'enc_label']]
LABELS

In [ ]:
# Defining some key variables/configurations
MAX_LEN = 512
TRAIN_BATCH_SIZE = 16
VALID_BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 1e-05


In [ ]:
class Triage(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.len = len(dataframe)
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, index):
        sentence = str(self.data.sentence[index])
        sentence = " ".join(sentence.split())
        inputs = self.tokenizer.encode_plus(
            sentence,
            None,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            return_token_type_ids=True,
            truncation=True
        )
        ids = inputs['input_ids']
        mask = inputs['attention_mask']

        if 'enc_label' not in self.data:
            return {
            'ids': torch.tensor(ids, dtype=torch.long),
            'mask': torch.tensor(mask, dtype=torch.long)
            }

        return {
            'ids': torch.tensor(ids, dtype=torch.long),
            'mask': torch.tensor(mask, dtype=torch.long),
            'targets': torch.tensor(self.data.enc_label[index], dtype=torch.long)
        }

    def __len__(self):
        return self.len

Choose an appropriate name for saving your train and test datasets

In [ ]:
from sklearn.model_selection import train_test_split

#Train/Test split
train_indices, test_indices = train_test_split(list(range(len(df.enc_label))), test_size=0.2, stratify=df.enc_label)
train_dataset = df.copy().drop(test_indices).reset_index(drop=True)
test_dataset = df.copy().drop(train_indices).reset_index(drop=True)

test_dataset.to_csv('test_dataset_mixed_new.csv') #Change name here
train_dataset.to_csv('train_dataset_mixed_new.csv')

print("FULL Dataset: {}".format(df.shape))
print("TRAIN Dataset: {}".format(train_dataset.shape))
print("TEST Dataset: {}".format(test_dataset.shape))

training_set = Triage(train_dataset, tokenizer, MAX_LEN)
testing_set = Triage(test_dataset, tokenizer, MAX_LEN)

train_params = {'batch_size': TRAIN_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }

test_params = {'batch_size': VALID_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }

training_loader = DataLoader(training_set, **train_params)
testing_loader = DataLoader(testing_set, **test_params)

In [ ]:
# Creating the customized model, by adding a drop out and a dense layer on top of distil bert to get the final output for the model.

class SecBERTClass(torch.nn.Module):
    def __init__(self, pretrained_model_name: str, num_classes: int = None, dropout: float = 0.3):
        super().__init__()
        config = BertConfig.from_pretrained(pretrained_model_name, output_hidden_states=True)
        self.model = AutoModel.from_pretrained(pretrained_model_name, config=config) #picking only the main body of the model
        self.pre_classifier = torch.nn.Linear(768, 768)
        self.dropout = torch.nn.Dropout(dropout)
        self.classifier = torch.nn.Linear(768, num_classes)

    def forward(self, input_ids, attention_mask):
        output_1 = self.model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
        hidden_state = output_1[0]
        pooler = hidden_state[:, 0]
        pooler = self.pre_classifier(pooler)
        pooler = torch.nn.ReLU()(pooler)
        pooler = self.dropout(pooler)
        output = self.classifier(pooler)
        return output



In [ ]:
#LOAD
model = SecBERTClass("jackaduma/SecBERT", LABELS)

In [ ]:
model.to(device)

In [ ]:
len(train_dataset)

In [ ]:
# Creating the loss function and optimizer
loss_function = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params =  model.parameters(), lr=LEARNING_RATE)

# Function to calcuate the accuracy of the model (not used)
def calcuate_accu(big_idx, targets):
    n_correct = (big_idx==targets).sum().item()
    return n_correct

In [ ]:
torch.cuda.empty_cache()

# Defining the training function on the 80% of the dataset for tuning the distilbert model

def train(epoch):

    examples = len(train_dataset)
    losses = [None] * len(training_loader)
    model.train()
    #loop = tqdm(enumerate(training_loader), total=len(training_loader), leave=False)
    for i, data in enumerate(training_loader, 0):
        #print(i)
        ids = data['ids'].to(device, dtype = torch.long)
        mask = data['mask'].to(device, dtype = torch.long)
        targets = data['targets'].to(device, dtype = torch.long)

        outputs = model(ids, mask)
        loss = loss_function(outputs, targets)

        losses[i] = loss.item()
        optimizer.zero_grad()
        loss.backward()
        # # When using GPU
        optimizer.step()

    print(f"Cost at epoch {epoch} is {sum(losses)/len(losses):.5f}")
    return


for epoch in range(EPOCHS):
    train(epoch)

In [ ]:
def check_accuracy(loader, model):

    num_correct = 0
    num_samples = 0
    model.eval()

    with torch.no_grad():
      for i, data in enumerate(loader, 0):
          x = data['ids'].to(device, dtype = torch.long)
          mask = data['mask'].to(device, dtype = torch.long)
          y = data['targets'].to(device, dtype = torch.long)

          scores = model(x, mask)
          _, predictions = scores.max(1)
          num_correct += (predictions == y).sum()
          num_samples += predictions.size(0)

      print(
          f"Got {num_correct} / {num_samples} with accuracy {float(num_correct)/float(num_samples)*100:.2f}"
      )



Save the model for further tests

In [ ]:
import os
import torch
import joblib
from transformers import AutoTokenizer

# Saving artifacts and downloading them
os.makedirs("mitre_model", exist_ok=True)


torch.save(model.state_dict(), "mitre_model/model.pt")

tokenizer.save_pretrained("mitre_model")

joblib.dump(encoder, "mitre_model/label_encoder.pkl")
print("Saved encoder label count:", len(encoder.classes_))

with open("mitre_model/config.txt", "w") as f:
    f.write(f"num_labels={LABELS}\n")
    f.write("model_base=jackaduma/SecBERT\n")

import shutil
from google.colab import files

shutil.make_archive("mitre_model", 'zip', "mitre_model")
files.download("mitre_model.zip")


In [ ]:
#Loading tokenizer before, encoder model initialize model and weights before Final accuracy check
tokenizer = AutoTokenizer.from_pretrained("mitre_model")

import joblib
encoder = joblib.load("mitre_model/label_encoder.pkl")

model = SecBERTClass("jackaduma/SecBERT", num_classes=LABELS)
model.load_state_dict(torch.load("mitre_model/model.pt", map_location=torch.device('cpu')))
model.eval()


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

#Final accuracy check
check_accuracy(testing_loader, model)
